# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [ ]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
from dotenv import load_dotenv
import os
import time

# API key disimpan di file .env (root project), lihat .env.example
load_dotenv('../.env')
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

EMBEDDING_MODEL = os.environ.get('EMBEDDING_MODEL', 'gemini-embedding-001')
LLM_MODEL = os.environ.get('LLM_MODEL', 'gemini-flash-latest')

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

In [ ]:
# Cek struktur data
print('Kolom:', list(df.columns))
print('Jumlah kategori:', df['category'].nunique())
print('Missing value:', df.isna().sum().sum())
print()
print(df['category'].value_counts())

## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

Kategori juga ikut digabungkan supaya topik FAQ lebih jelas.

In [ ]:
df['content'] = 'Kategori: ' + df['category'] + '\nQ: ' + df['question'] + '\nA: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

Catatan: model `text-embedding-004` sudah tidak tersedia, jadi diganti dengan `gemini-embedding-001`. Semua dokumen dikirim sekaligus dalam satu request (batch) supaya tidak terkena rate limit.

In [ ]:
def get_embedding(text, task_type='RETRIEVAL_QUERY'):
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

# Embedding semua FAQ sekaligus (batch)
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=documents,
    config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT')
)
embeddings = [e.values for e in result.embeddings]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)
print('Jumlah entri FAQ:', len(df))
print('Sesuai?', index.ntotal == len(df))

# Simpan index ke file
os.makedirs('../vector_store', exist_ok=True)
faiss.write_index(index, '../vector_store/faq_index.faiss')
print('Index disimpan ke vector_store/faq_index.faiss')

## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

Distance yang dipakai adalah L2, jadi **semakin kecil distance, semakin mirip** query dengan FAQ.

In [ ]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'faq_id': df.iloc[idx]['faq_id'],
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': round(float(distances[0][rank]), 4)
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
pd.DataFrame(test_results)[['rank', 'faq_id', 'category', 'question', 'distance']]

## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [ ]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"[FAQ #{r['faq_id']}]\nPertanyaan: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''PERSONA:
Kamu adalah asisten customer service e-commerce yang ramah dan membantu. Gunakan Bahasa Indonesia yang sopan dan mudah dipahami.

TASK:
Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ pada bagian CONTEXT. Jangan menggunakan pengetahuan di luar FAQ dan jangan mengarang informasi.
Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan pengguna untuk menghubungi customer service.

CONTEXT (FAQ relevan):
{context}

FORMAT:
- Jawab singkat dan jelas, maksimal 4 kalimat.
- Di akhir jawaban tuliskan sumbernya, contoh: "Sumber: FAQ #17". Jika informasi tidak ada di FAQ, tulis "Sumber: -".

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

# Lihat contoh prompt
print(build_prompt('apakah bisa bayar pakai cicilan?', semantic_search('apakah bisa bayar pakai cicilan?')))

## 7. Mengintegrasikan LLM

In [ ]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    # Coba ulang beberapa kali jika server Gemini sedang sibuk (error 503 / 429)
    for percobaan in range(5):
        try:
            response = client.models.generate_content(
                model=LLM_MODEL,  # sesuaikan versi model
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.3)
            )
            return response.text, search_results
        except Exception as e:
            if percobaan == 4:
                raise
            print(f'Server sibuk ({e.__class__.__name__}), coba lagi dalam 15 detik...')
            time.sleep(15)

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', f"FAQ #{s['faq_id']}", s['question'], '| distance:', s['distance'])

## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [ ]:
test_queries = [
    # Query yang jelas ada di FAQ
    'Metode pembayaran apa saja yang tersedia?',                          # harusnya FAQ #17
    'Bagaimana cara membatalkan pesanan?',                                # harusnya FAQ #3
    # Query parafrase / sinonim
    'Barang yang aku terima pecah dan kardusnya penyok, harus gimana?',  # harusnya FAQ #16
    'Kalau beli HP mahal bisa dicicil per bulan nggak?',                  # harusnya FAQ #20
    'Uang saya kapan balik kalau barangnya sudah saya retur?',           # harusnya FAQ #25
    # Query yang tidak ada di FAQ
    'Apakah ada toko offline yang bisa saya kunjungi di Bandung?',
    'Apakah saya bisa membayar menggunakan Bitcoin atau crypto?',
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Hasil retrieval:')
    for s in sources:
        print(f"  {s['rank']}. FAQ #{s['faq_id']} - {s['question']} (distance: {s['distance']})")
    print('---')

### Catatan Analisis Relevansi

_Tuliskan pengamatanmu di sini: query mana yang berhasil di-retrieve dengan baik, query mana yang kurang relevan hasilnya, dan bagaimana LLM merespons ketika informasi tidak tersedia di knowledge base._

(Diisi setelah notebook dijalankan.)